In [1]:
# CELL 1 — update path to your DDoS CSV file
import os, pandas as pd
csv_path = r"C:\Users\syeds\OneDrive\Documents\Desktop\CyberAI-NextGen-Defender\CyberAI-NextGen-Defender\data\ddos\ddos 2.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"CSV not found: {csv_path}. Put the file path here and rerun this cell.")

# quick preview
print("File exists. Preview first non-empty 3 rows:")
with open(csv_path, 'r', encoding='utf-8', errors='replace') as f:
    preview = []
    for line in f:
        if line.strip():
            preview.append(line.strip())
        if len(preview) >= 3:
            break
    for i, ln in enumerate(preview, 1):
        print(f"{i}: {ln[:400]}")
        
# print detected number of fields in first row
first_line = preview[0]
num_fields = len(first_line.split(","))
print("Detected fields (first row):", num_fields)


File exists. Preview first non-empty 3 rows:
1: 192.168.213.131,dnsresearch.ml,1636816571,True,carweb.dnsresearch.ml,6,1,2,3,2.584962500721156,0.5,0.3333333333333333,0.0,0.0,0,0.0,8.4,7.589466384404111,84000.0,0.0,2.5361427684638986,0.7121277581461345
2: 192.168.213.131,dnsresearch.ml,1636816571,True,lednot.dnsresearch.ml,6,1,2,3,2.584962500721156,0.5,0.3333333333333333,0.0,0.0,0,0.0,6,0.0,60000.0,0.0,2.3597995839823738,0.32851380756119675
3: 192.168.213.131,dnsresearch.ml,1636816571,True,webnot.dnsresearch.ml,6,1,2,3,2.584962500721156,0.5,0.3333333333333333,0.0,0.0,0,0.0,6,0.0,60000.0,0.0,2.393132917315707,0.33319635264813136
Detected fields (first row): 22


In [2]:
# CELL 2 — read into pandas with safe headers
import pandas as pd

# Adjust num_fields from previous cell if needed
num_fields = 22   # <-- update if CELL 1 printed different number

# human-friendly header list if data follows the structure we discussed
named_headers = [
    "src_ip","domain","timestamp","is_attack","target","protocol",
    "flag1","flag2","flag3","entropy","f1","f2","f3","f4","f5","f6","f7","f8","duration","f9","f10","f11"
]

if len(named_headers) != num_fields:
    # fallback to generic headers
    headers = [f"col_{i}" for i in range(num_fields)]
    print("Using generic headers:", headers[:6], "...")
else:
    headers = named_headers
    print("Using named headers.")

# load
df = pd.read_csv(csv_path, names=headers, header=None, engine="python")
print("Loaded shape:", df.shape)
display(df.head())
print("\nDtypes:")
print(df.dtypes)


Using named headers.
Loaded shape: (42301, 22)


,src_ip,domain,timestamp,is_attack,target,protocol,flag1,flag2,flag3,entropy,...,f3,f4,f5,f6,f7,f8,duration,f9,f10,f11
0,192.168.213.131,dnsresearch.ml,1636816571,True,carweb.dnsresearch.ml,6,1,2,3,2.584963,...,0.0,0.0,0.000000,0.000000,8.4,7.589466,84000.0,0.0,2.536143,0.712128
1,192.168.213.131,dnsresearch.ml,1636816571,True,lednot.dnsresearch.ml,6,1,2,3,2.584963,...,0.0,0.0,0.000000,0.000000,6.0,0.000000,60000.0,0.0,2.359800,0.328514
2,192.168.213.131,dnsresearch.ml,1636816571,True,webnot.dnsresearch.ml,6,1,2,3,2.584963,...,0.0,0.0,0.000000,0.000000,6.0,0.000000,60000.0,0.0,2.393133,0.333196
3,192.168.213.131,dnsresearch.ml,1636816571,True,cutnot.dnsresearch.ml,6,1,2,3,2.251629,...,0.0,0.0,0.000000,0.000000,6.0,0.000000,60000.0,0.0,2.439048,0.266111
4,192.168.213.131,dnsresearch.ml,1636816572,True,setpen.dnsresearch.ml,6,1,2,3,2.251629,...,0.0,0.0,0.111111,0.333333,6.0,0.000000,30000.0,0.0,2.405714,0.266679



Dtypes:
src_ip        object
domain        object
timestamp      int64
is_attack       bool
target        object
protocol       int64
flag1          int64
flag2          int64
flag3          int64
entropy      float64
f1           float64
f2           float64
f3           float64
f4           float64
f5           float64
f6           float64
f7           float64
f8           float64
duration     float64
f9           float64
f10          float64
f11          float64
dtype: object


In [3]:
# CELL 3 — fix types and map is_attack to 0/1
import numpy as np

# If timestamp is epoch, convert to datetime for readability (optional)
if 'timestamp' in df.columns:
    try:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
    except Exception:
        pass

# If is_attack is boolean or string, map to 1/0
if df['is_attack'].dtype == object:
    df['is_attack'] = df['is_attack'].map({'True':1, 'False':0, 'true':1, 'false':0}).fillna(df['is_attack'])

# If still not numeric, try convert
df['is_attack'] = pd.to_numeric(df['is_attack'], errors='coerce').fillna(1).astype(int)

print("is_attack value counts:")
print(df['is_attack'].value_counts())


is_attack value counts:
is_attack
1    42301
Name: count, dtype: int64


In [4]:
# CELL 4 — generate synthetic normal traffic if needed
import numpy as np

if df['is_attack'].nunique() == 1:
    print("Dataset contains only one class; creating synthetic normal traffic to balance.")
    num = len(df)
    df_normal = df.copy()
    numeric_cols = [c for c in df.columns if df[c].dtype in [np.float64, np.int64] and c!='is_attack']
    # add gentle noise to numeric features to simulate normal patterns
    for col in numeric_cols:
        arr = df_normal[col].astype(float).values
        noise = np.random.uniform(0.6, 0.95, size=arr.shape)
        df_normal[col] = arr * noise
    df_normal['is_attack'] = 0
    df_combined = pd.concat([df, df_normal], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
    df = df_combined
    print("Combined shape:", df.shape, "value counts:", df['is_attack'].value_counts().to_dict())
else:
    print("Dataset already contains multiple classes.")


Dataset contains only one class; creating synthetic normal traffic to balance.
Combined shape: (84602, 22) value counts: {1: 42301, 0: 42301}


In [6]:
# --- Encode non-numeric columns, save encoders, then scale ---
import os
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler

os.makedirs("models", exist_ok=True)

# identify object / non-numeric columns in X_train
non_numeric_cols = [c for c in X_train.columns if not pd.api.types.is_numeric_dtype(X_train[c])]
print("Non-numeric columns to encode:", non_numeric_cols)

# create dict to hold encoders
encoders = {}

# Apply LabelEncoder columnwise (fit on train, transform both train/test)
for col in non_numeric_cols:
    le = LabelEncoder()
    # fill missing with a placeholder string to avoid errors
    X_train[col] = X_train[col].fillna("##MISSING##").astype(str)
    X_test[col]  = X_test[col].fillna("##MISSING##").astype(str)
    le.fit(pd.concat([X_train[col], X_test[col]], axis=0).astype(str))
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))
    encoders[col] = le
    print(f"Encoded {col}: classes={len(le.classes_)}")

# Save encoders for later inference
joblib.dump(encoders, "models/ddos_label_encoders.joblib")
print("Saved label encoders -> models/ddos_label_encoders.joblib")

# Now scale numeric-only DataFrames (all columns numeric now)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, "models/ddos_scaler.joblib")
print("Saved scaler -> models/ddos_scaler.joblib")

# Quick sanity prints
import numpy as np
print("Scaled shapes:", X_train_scaled.shape, X_test_scaled.shape)
print("Example scaled row (first train row):", np.round(X_train_scaled[0,:6], 4))


Non-numeric columns to encode: ['src_ip', 'domain', 'target']
Encoded src_ip: classes=2
Encoded domain: classes=1
Encoded target: classes=32852
Saved label encoders -> models/ddos_label_encoders.joblib
Saved scaler -> models/ddos_scaler.joblib
Scaled shapes: (67681, 20) (16921, 20)
Example scaled row (first train row): [ 0.0272  0.      0.3444 -0.4953 -0.4926 -0.4737]


In [7]:
# CELL 5 — prepare X and y, encode categorical columns (safe)
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# df should already exist (from your previous synthetic combine step)
print("df shape:", df.shape)

# Drop timestamp (optional) and any unwanted columns
drop_cols = []
if 'timestamp' in df.columns:
    drop_cols.append('timestamp')

# pick categorical columns to encode (commonly present)
cat_cols = [c for c in ['src_ip','domain','target'] if c in df.columns]
print("Categorical cols detected:", cat_cols)

# Ensure no 'is_attack' in X
X = df.drop(columns=drop_cols + ['is_attack'], errors='ignore').copy()
y = df['is_attack'].astype(int).copy()

# Label encode categorical columns (fit on whole df to avoid unseen later in demo)
encoders = {}
from sklearn.preprocessing import LabelEncoder
for c in cat_cols:
    le = LabelEncoder()
    X[c] = X[c].fillna("##MISSING##").astype(str)
    le.fit(X[c])
    X[c] = le.transform(X[c])
    encoders[c] = le
    print(f"Encoded {c}: {len(le.classes_)} classes")

# Save column order (important)
feature_names = list(X.columns)
print("Feature count:", len(feature_names))
print("Feature sample (first row):")
display(X.head(1))


df shape: (84602, 22)
Categorical cols detected: ['src_ip', 'domain', 'target']
Encoded src_ip: 2 classes
Encoded domain: 1 classes
Encoded target: 32852 classes
Feature count: 20
Feature sample (first row):


,src_ip,domain,target,protocol,flag1,flag2,flag3,entropy,f1,f2,f3,f4,f5,f6,f7,f8,duration,f9,f10,f11
0,1,0,11108,25.0,2.0,13.0,5.0,3.669275,0.2,0.52,0.0,0.0,0.444444,0.527046,25.0,0.0,50000.0,0.0,3.654666,0.142906


In [8]:
# CELL 6 — split and save encoders
import joblib, os
from sklearn.model_selection import train_test_split

os.makedirs("models", exist_ok=True)
joblib.dump(encoders, "models/ddos_label_encoders.joblib")
print("Saved encoders -> models/ddos_label_encoders.joblib")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train label counts:\\n", y_train.value_counts())
print("Test  label counts:\\n", y_test.value_counts())


Saved encoders -> models/ddos_label_encoders.joblib
Train shape: (67681, 20) Test shape: (16921, 20)
Train label counts:\n is_attack
1    33841
0    33840
Name: count, dtype: int64
Test  label counts:\n is_attack
0    8461
1    8460
Name: count, dtype: int64


In [9]:
# CELL 7 — scale and save scaler
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

joblib.dump(scaler, "models/ddos_scaler.joblib")
print("Saved scaler -> models/ddos_scaler.joblib")
print("Scaled shapes:", X_train_scaled.shape, X_test_scaled.shape)


Saved scaler -> models/ddos_scaler.joblib
Scaled shapes: (67681, 20) (16921, 20)


In [11]:
# CELL 8 — train XGBoost, evaluate, save model & summary (corrected)
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import json

xgb_model = xgb.XGBClassifier(
    n_estimators=250, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, use_label_encoder=False,
    eval_metric='logloss', random_state=42
)

# Fit without early stopping to avoid version issues
xgb_model.fit(X_train_scaled, y_train)

# Predict
y_pred = xgb_model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# save model
import os
os.makedirs("models", exist_ok=True)
xgb_model.save_model("models/ddos_xgb.json")
print("Saved XGBoost model -> models/ddos_xgb.json")

# feature importance summary
try:
    fmap = xgb_model.get_booster().get_score(importance_type='gain')
    imp = sorted(fmap.items(), key=lambda x: x[1], reverse=True)
    top_imp = imp[:20]
    print("Top feature importance (gain) - top 20:")
    for feat, gain in top_imp:
        print(f"  {feat}: {gain:.4f}")
except Exception as e:
    print("Could not compute feature importance:", e)

# save artifact summary
summary = {
    "model":"models/ddos_xgb.json",
    "scaler":"models/ddos_scaler.joblib",
    "encoders":"models/ddos_label_encoders.joblib",
    "num_features": X_train.shape[1],
    "train_rows": int(X_train.shape[0]),
    "test_rows": int(X_test.shape[0])
}
with open("models/ddos_artifacts_summary.json","w") as f:
    json.dump(summary, f, indent=2)
print("Saved summary -> models/ddos_artifacts_summary.json")


C:\Users\syeds\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [03:33:10] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 0.9998227055138585
              precision    recall  f1-score   support

           0     1.0000    0.9996    0.9998      8461
           1     0.9996    1.0000    0.9998      8460

    accuracy                         0.9998     16921
   macro avg     0.9998    0.9998    0.9998     16921
weighted avg     0.9998    0.9998    0.9998     16921

Confusion matrix:
 [[8458    3]
 [   0 8460]]
Saved XGBoost model -> models/ddos_xgb.json
Top feature importance (gain) - top 20:
  f4: 710.7155
  f18: 289.9900
  f8: 201.3168
  f6: 155.1151
  f7: 87.7553
  f10: 36.1607
  f3: 33.7022
  f13: 28.2714
  f12: 21.4694
  f14: 20.9443
  f19: 13.4650
  f5: 9.1850
  f11: 7.9529
  f9: 7.9301
  f15: 5.0047
  f16: 2.3001
  f2: 0.3575
Saved summary -> models/ddos_artifacts_summary.json


In [13]:
#defender
import numpy as np
import pandas as pd

# Example using XGBoost predictions
y_prob = xgb_model.predict_proba(X_test_scaled)[:,1]

def defender_action(prob):
    if prob >= 0.9:
        return "Block traffic"
    elif prob >= 0.6:
        return "Monitor traffic"
    else:
        return "Allow traffic"

df_defense = pd.DataFrame({
    "predicted_label": y_pred,
    "prediction_prob": y_prob,
    "defense_action": [defender_action(p) for p in y_prob]
})

df_defense.head(10)
df_defense.to_csv("models/ddos_defense_results.csv", index=False)
print("Defender results saved -> models/ddos_defense_results.csv")


Defender results saved -> models/ddos_defense_results.csv


In [14]:
import pandas as pd
res = pd.read_csv("models/ddos_defense_results.csv")
print("Rows:", len(res))
print("Action counts:")
print(res['defense_action'].value_counts())
display(res.head(10))


Rows: 16921
Action counts:
defense_action
Allow traffic      8460
Block traffic      8460
Monitor traffic       1
Name: count, dtype: int64


,predicted_label,prediction_prob,defense_action
0,0,0.000064,Allow traffic
1,0,0.001132,Allow traffic
2,0,0.000012,Allow traffic
3,0,0.000012,Allow traffic
4,0,0.000041,Allow traffic
5,0,0.000023,Allow traffic
6,1,0.999863,Block traffic
7,0,0.000124,Allow traffic
8,1,0.999843,Block traffic
9,1,0.999964,Block traffic


In [16]:
import pandas as pd
import joblib
import json
import xgboost as xgb
import numpy as np

# Load scaler and encoders
scaler = joblib.load("models/ddos_scaler.joblib")
encoders = joblib.load("models/ddos_label_encoders.joblib")

# Load XGBoost model
xgb_model = xgb.XGBClassifier()
xgb_model.load_model("models/ddos_xgb.json")

# Load test data (assume X_test_scaled is ready)
X_test_scaled = scaler.transform(X_test)  # or load from your preprocessing
y_test = y_test  # your test labels

# Predict probabilities
y_prob = xgb_model.predict_proba(X_test_scaled)[:,1]
y_pred = (y_prob > 0.5).astype(int)

# Map predicted label to defense actions
defense_action = []
for prob, pred in zip(y_prob, y_pred):
    if prob > 0.9:
        defense_action.append("block_ip")
    elif prob > 0.6:
        defense_action.append("rate_limit")
    else:
        defense_action.append("allow")

# Save results
results = pd.DataFrame({
    "predicted_label": y_pred,
    "prediction_prob": y_prob,
    "defense_action": defense_action
})
results.to_csv("models/ddos_defense_results.csv", index=False)

# Optional: action log (for defender history)
action_log = pd.DataFrame({
    "ip": X_test["src_ip"],  # replace with your test source IP column
    "predicted_label": y_pred,
    "prob": y_prob,
    "defense_action": defense_action
})
action_log.to_csv("models/defender_action_log.csv", index=False)

print("Defender results saved -> models/ddos_defense_results.csv")
print("Action log saved -> models/defender_action_log.csv")


Defender results saved -> models/ddos_defense_results.csv
Action log saved -> models/defender_action_log.csv


In [18]:
import pandas as pd, os

# load files (defense results and action log)
res_path = "models/ddos_defense_results.csv"
alog_path = "models/defender_action_log.csv"

print("Exists:", res_path, "->", os.path.exists(res_path))
print("Exists:", alog_path, "->", os.path.exists(alog_path))

if os.path.exists(res_path):
    res = pd.read_csv(res_path)
    print("ddos_defense_results.csv rows:", len(res))
    print(res['defense_action'].value_counts())
    display(res.head(10))

if os.path.exists(alog_path):
    alog = pd.read_csv(alog_path)
    print("\\ndefender_action_log.csv rows:", len(alog))
    display(alog.tail(12))


Exists: models/ddos_defense_results.csv -> True
Exists: models/defender_action_log.csv -> True
ddos_defense_results.csv rows: 16921
defense_action
allow         8460
block_ip      8460
rate_limit       1
Name: count, dtype: int64


,predicted_label,prediction_prob,defense_action
0,0,0.000064,allow
1,0,0.001132,allow
2,0,0.000012,allow
3,0,0.000012,allow
4,0,0.000041,allow
5,0,0.000023,allow
6,1,0.999863,block_ip
7,0,0.000124,allow
8,1,0.999843,block_ip
9,1,0.999964,block_ip


\ndefender_action_log.csv rows: 16921


,ip,predicted_label,prob,defense_action
16909,1,0,0.000012,allow
16910,1,0,0.000112,allow
16911,1,0,0.000009,allow
16912,1,0,0.000039,allow
16913,1,1,0.999912,block_ip
16914,1,1,0.999782,block_ip
16915,1,0,0.000031,allow
16916,1,0,0.000011,allow
16917,1,1,0.999860,block_ip
16918,1,1,0.999681,block_ip


In [20]:
import pandas as pd
import csv
import os

alog_path = "models/defender_action_log.csv"
blocklist_path = "models/ddos_blocklist.csv"

alog = pd.read_csv(alog_path)

# Filter rows where action is block_ip
blocks = alog[alog['defense_action'] == 'block_ip']

# Write blocklist
with open(blocklist_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ip','reason','score'])  # header
    for _, row in blocks.iterrows():
        ip = row['ip']
        score = row['prob'] if 'prob' in row else None
        writer.writerow([ip, 'auto-block', score])

print(f"Blocklist created with {len(blocks)} entries -> {blocklist_path}")

# Quick check
bdf = pd.read_csv(blocklist_path)
display(bdf.head(10))


Blocklist created with 8460 entries -> models/ddos_blocklist.csv


,ip,reason,score
0,1,auto-block,0.999863
1,1,auto-block,0.999843
2,1,auto-block,0.999964
3,1,auto-block,0.999938
4,1,auto-block,0.999806
5,1,auto-block,0.999972
6,1,auto-block,0.999929
7,1,auto-block,0.999903
8,1,auto-block,0.999463
9,1,auto-block,0.999956


In [21]:
import pandas as pd, os, shutil
blk = "models/ddos_blocklist.csv"
if os.path.exists(blk):
    bdf = pd.read_csv(blk)
    print("Blocked IPs count (rows):", len(bdf))
    # show top 10 by frequency if ip column exists
    if 'ip' in bdf.columns:
        print("Top blocked IPs:")
        display(bdf['ip'].value_counts().head(10))
    display(bdf.head(12))
else:
    print("Blocklist file not found at", blk)

# Backup important files to a backup folder
os.makedirs("backup_models", exist_ok=True)
for p in ["models/ddos_xgb.json","models/ddos_scaler.joblib","models/ddos_label_encoders.joblib","models/ddos_defense_results.csv","models/defender_action_log.csv","models/ddos_blocklist.csv"]:
    if os.path.exists(p):
        shutil.copy(p, "backup_models")
        print("Backed up:", p)
    else:
        print("Missing (not backed up):", p)
print("Backup done -> backup_models/")


Blocked IPs count (rows): 8460
Top blocked IPs:


ip
1    8457
0       3
Name: count, dtype: int64

,ip,reason,score
0,1,auto-block,0.999863
1,1,auto-block,0.999843
2,1,auto-block,0.999964
3,1,auto-block,0.999938
4,1,auto-block,0.999806
5,1,auto-block,0.999972
6,1,auto-block,0.999929
7,1,auto-block,0.999903
8,1,auto-block,0.999463
9,1,auto-block,0.999956


Backed up: models/ddos_xgb.json
Backed up: models/ddos_scaler.joblib
Backed up: models/ddos_label_encoders.joblib
Backed up: models/ddos_defense_results.csv
Backed up: models/defender_action_log.csv
Backed up: models/ddos_blocklist.csv
Backup done -> backup_models/


In [1]:
import os
import joblib
import xgboost as xgb
from xgboost import XGBClassifier
import pandas as pd
import numpy as np
import shap
from sklearn.feature_extraction.text import TfidfVectorizer

# -------------------------------
# 1️⃣ Set threat
# -------------------------------
THREAT_NAME = "ddos"  

# -------------------------------
# 2️⃣ Paths
# -------------------------------
model_json = f"models/{THREAT_NAME}_xgb.json"
pkl_model = f"models/{THREAT_NAME}_xgboost_model.pkl"
defense_csv = f"models/{THREAT_NAME}_defense_results.csv"
vectorizer_path = f"models/{THREAT_NAME}_tfidf_vectorizer.pkl"
explain_log = f"models/{THREAT_NAME}_explain_log.csv"

# -------------------------------
# 3️⃣ Convert JSON → PKL if not exists
# -------------------------------
if not os.path.exists(pkl_model) and os.path.exists(model_json):
    booster = xgb.Booster()
    booster.load_model(model_json)
    clf = XGBClassifier()
    clf._Booster = booster
    clf._le = None
    joblib.dump(clf, pkl_model)
    print(f"Saved PKL model -> {pkl_model}")
else:
    print(f"PKL model exists -> {pkl_model}")

# -------------------------------
# 4️⃣ Load model and defense log
# -------------------------------
model = joblib.load(pkl_model)
defense_results = pd.read_csv(defense_csv)
print("Columns in defense CSV:", defense_results.columns.tolist())

# -------------------------------
# 5️⃣ Determine text column
# -------------------------------
text_column = None
for col in ["text", "text_sample"]:
    if col in defense_results.columns:
        text_column = col
        break

if text_column:
    defense_results.rename(columns={text_column: "text"}, inplace=True)
else:
    print("⚠️ No text column found. SHAP will use IDs instead.")

# -------------------------------
# 6️⃣ Filter quarantined samples
# -------------------------------
quarantine_samples = defense_results[defense_results["defense_action"] == "quarantine"].copy()
print(f"Total quarantined samples: {len(quarantine_samples)}")

# -------------------------------
# 7️⃣ TF-IDF Vectorizer
# -------------------------------
if os.path.exists(vectorizer_path):
    vectorizer = joblib.load(vectorizer_path)
else:
    vectorizer = TfidfVectorizer(max_features=3000)
    if "text" in defense_results.columns:
        vectorizer.fit(defense_results["text"].astype(str).tolist())
        joblib.dump(vectorizer, vectorizer_path)
        print(f"Saved new TF-IDF vectorizer -> {vectorizer_path}")
    else:
        vectorizer = None

if len(quarantine_samples) > 0 and vectorizer:
    X_features = vectorizer.transform(quarantine_samples["text"].astype(str).tolist())
    feature_names = np.array(vectorizer.get_feature_names_out())
else:
    X_features = None
    feature_names = None

# -------------------------------
# 8️⃣ SHAP Explainability
# -------------------------------
explain_data = []

if X_features is not None:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_features)

    for i in range(len(quarantine_samples)):
        shap_row = shap_values[i].flatten() if isinstance(shap_values, np.ndarray) else shap_values[i].toarray().flatten()
        top_indices = np.argsort(np.abs(shap_row))[-10:][::-1]
        top_feats = feature_names[top_indices]
        top_vals = shap_row[top_indices]
        top_pairs = [f"{f} ({round(v,3)})" for f,v in zip(top_feats, top_vals)]

        row_dict = {"index": quarantine_samples.index[i]}
        if "text" in quarantine_samples.columns:
            row_dict["text"] = quarantine_samples.iloc[i]["text"][:120] + ("..." if len(quarantine_samples.iloc[i]["text"]) > 120 else "")
        else:
            row_dict["sample_id"] = quarantine_samples.iloc[i].get("ip_or_id", quarantine_samples.index[i])

        row_dict["top_contributing_features"] = ", ".join(top_pairs)
        explain_data.append(row_dict)
else:
    print("⚠️ No quarantined samples or text data. Explain log will be empty.")

# -------------------------------
# 9️⃣ Save Explainability CSV
# -------------------------------
os.makedirs("models", exist_ok=True)
explain_df = pd.DataFrame(explain_data)
explain_df.to_csv(explain_log, index=False)
print(f"Explainability log saved -> {explain_log}")
print(f"✅ Completed: {THREAT_NAME}")


Saved PKL model -> models/ddos_xgboost_model.pkl
Columns in defense CSV: ['predicted_label', 'prediction_prob', 'defense_action']
⚠️ No text column found. SHAP will use IDs instead.
Total quarantined samples: 0
⚠️ No quarantined samples or text data. Explain log will be empty.
Explainability log saved -> models/ddos_explain_log.csv
✅ Completed: ddos
